In [2]:
# (given) imports, device, seed, and a tiny helper for the check cells
import time
import torch
import torch.nn as nn
from torch.nn import functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(1337)
print('device:', device)

def check(name, cond, hint=''):
    assert bool(cond), f'FAIL: {name}. {hint}'
    print(f'PASS: {name}')

device: cpu


In [3]:
# hyperparameters for Parts 1-6 (small on purpose). Part 7 scales them up.
batch_size = 32    # how many independent sequences we process in parallel
block_size = 8     # maximum context length for predictions
n_embd = 32        # embedding dimension
n_head = 4         # number of attention heads
n_layer = 3        # number of transformer blocks
dropout = 0.0      # we switch dropout on in Part 7

In [4]:
# (given) read the dataset and look at it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()
print('length of dataset in characters:', len(text))
print(text[:300])

length of dataset in characters: 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


In [5]:
chars = sorted(set(text))
vocab_size = len(chars)

stoi = {c : i for i, c in enumerate(chars)} 
itos = {i : c for i, c in enumerate(chars)}

def encode(s):  # string -> list of integers
    return [stoi[c] for c in s]

def decode(l):  # list of integers -> string
    return ''.join([itos[i] for i in l])

encoded_data = encode(text)
data = torch.tensor(encoded_data, dtype=torch.long)

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

block_size = 8
x = train_data[:block_size]
y = train_data[1:block_size+1]


In [6]:
def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, size=(batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x,y = x.to(device), y.to(device)
    return x, y

torch.manual_seed(1337)
xb, yb = get_batch("train")

In [8]:
num_embeddings, embedding_dim = vocab_size, vocab_size

emb = nn.Embedding(num_embeddings, embedding_dim)
idx = torch.tensor([[1, 5, 7], [2, 2, 9]])
out = emb(idx)

In [ ]:
from typing import Any


class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size) -> None:
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    def forward(self, sequences, targets=None): 
        logits = self.token_embedding_table(sequences)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(-1, C)
            targets = targets.view(-1)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits = self.token_embedding_table(idx)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            new_sequence = torch.cat((idx, next_token), dim=1)
        return new_sequence

torch.manual_seed(1337)
model = BigramLanguageModel(vocab_size)
logits, loss = model(xb, yb)

logits:  torch.Size([256, 65])
loss:  torch.Size([])


In [ ]:
expected_init_loss = -torch.log(torch.tensor(1.0 / vocab_size)).item()
